In [1]:
import pandas as pd
import pickle
import numpy as np 
import random
from tqdm import tqdm
from pathlib import Path
import re

In [2]:
np.random.seed(0)


In [3]:
# file_out_dir = list(Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/').glob('*dataset*.pdpkl'))
# Import timit data - this df contains all the relevant signals, labels and data 
path = '/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl'

timit_excerpts = pd.read_pickle(path)


In [4]:
## Made in timit_screen.ipynb on local desktop 

bad_sentences = ['sx74', 'sx144', 'sx376', 'si615', 'si670', 'si688', 'si693',
       'si917', 'si926', 'si927', 'si930', 'si931', 'si1078', 'si1102',
       'si1106', 'si1241', 'si1616', 'si1739', 'si1752', 'si1758',
       'si1767', 'si1933', 'si2020', 'si2040', 'si2064', 'si2112',
       'si2134', 'si2204', 'si2284']


timit_excerpts = timit_excerpts[~timit_excerpts.sentence_id.isin(bad_sentences)]

In [5]:
## Normalize all audio signals 
all_signals = np.stack(timit_excerpts.signal)

all_signals = all_signals - all_signals.mean(1).reshape(-1,1) # de-mean rows 


per_signal_rms = np.sqrt(np.mean(np.power(all_signals, 2), axis=1)).reshape(-1,1) # per signal rms 


all_signals = all_signals * 0.1 / per_signal_rms # normalize all signals to rms of 0.02 or ~60dB

timit_excerpts['signal'] = [signal for signal in all_signals] # update signal column

In [6]:
timit_excerpts.columns

Index(['word', '_full_dataset_index', 'source', 'speaker', 'sr',
       'signal_length', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'data_split', 'signal', 'word_int'],
      dtype='object')

In [7]:
timit_excerpts = timit_excerpts.rename(columns={"_full_dataset_index":"_original_timit_index"})

In [8]:
# parse target & rest 

target_timit = timit_excerpts[timit_excerpts.word_int != 0]
# target_sentences = target_timit.sentence_id.unique()

# filter cues 
cue_timit = timit_excerpts[timit_excerpts.word_int == 0]
# cue_timit = timit_excerpts[(timit_excerpts.word_int == 0) & (~timit_excerpts.sentence_id.str.contains('1|2'))]


In [10]:
next_pow_2 = lambda x: int(pow(2, np.ceil(np.log2(x))))

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize(wav, new_rms=0.1, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor

## Define windowing function - will apply a cosine ramp to the start and end of a signal

In [11]:
## Precompute spectrum of all signals for speech shaped noise 

all_foregrounds = np.stack(target_timit['signal'])
n_x = all_foregrounds.shape[-1]
n_fft = next_pow_2(n_x)
X = np.fft.rfft(all_foregrounds, next_pow_2(n_fft))
X = X.mean(0)

def generate_speech_shaped_noise(X=X):
    # Randomize phase.
    noise_mag = np.abs(X) * np.exp(
        2 * np.pi * 1j * np.random.random(X.shape[-1]))
    noise = np.real(np.fft.irfft(noise_mag, n_fft))
    ssn = noise[:n_x]
    return ssn

In [12]:
## Make trial conditions 
import itertools


# setup design
snrs = [-6, -3, 0, 3] 

single_talker_conds = ['m', 'f']

two_talker_conds = list(itertools.combinations_with_replacement(single_talker_conds,2))
four_talker_conds = list(itertools.combinations_with_replacement(single_talker_conds,4))

distractor_conds = single_talker_conds + two_talker_conds + four_talker_conds + ['ssn']
condition_pairs = list(itertools.product(snrs, distractor_conds))
condition_pairs.append((-9,'m')) # only want -9dB for 1 distractor
condition_pairs.append((-9,'f')) # only want -9dB for 1 distractor

print(condition_pairs)

# file_out_dir = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes_mturk/')

[(-6, 'm'), (-6, 'f'), (-6, ('m', 'm')), (-6, ('m', 'f')), (-6, ('f', 'f')), (-6, ('m', 'm', 'm', 'm')), (-6, ('m', 'm', 'm', 'f')), (-6, ('m', 'm', 'f', 'f')), (-6, ('m', 'f', 'f', 'f')), (-6, ('f', 'f', 'f', 'f')), (-6, 'ssn'), (-3, 'm'), (-3, 'f'), (-3, ('m', 'm')), (-3, ('m', 'f')), (-3, ('f', 'f')), (-3, ('m', 'm', 'm', 'm')), (-3, ('m', 'm', 'm', 'f')), (-3, ('m', 'm', 'f', 'f')), (-3, ('m', 'f', 'f', 'f')), (-3, ('f', 'f', 'f', 'f')), (-3, 'ssn'), (0, 'm'), (0, 'f'), (0, ('m', 'm')), (0, ('m', 'f')), (0, ('f', 'f')), (0, ('m', 'm', 'm', 'm')), (0, ('m', 'm', 'm', 'f')), (0, ('m', 'm', 'f', 'f')), (0, ('m', 'f', 'f', 'f')), (0, ('f', 'f', 'f', 'f')), (0, 'ssn'), (3, 'm'), (3, 'f'), (3, ('m', 'm')), (3, ('m', 'f')), (3, ('f', 'f')), (3, ('m', 'm', 'm', 'm')), (3, ('m', 'm', 'm', 'f')), (3, ('m', 'm', 'f', 'f')), (3, ('m', 'f', 'f', 'f')), (3, ('f', 'f', 'f', 'f')), (3, 'ssn'), (-9, 'm'), (-9, 'f')]


In [13]:
def update_dict(dict_to_update, dict_with_vals):
    for key,value in dict_with_vals.items():
        if key not in dict_to_update:
            dict_to_update[key] = [value]
        else:
            dict_to_update[key].append(value)
            

In [14]:
# Generate trial by trial stim    
trial_data = {'signal':[],
              'speaker': [],
              'speaker_sex': [],
              'sentence_type': [],
              'word_int': [],
              'mixture_signal':[],
              'cue_signal': [],
              'cue_speaker': [],
              'cue_word': [],
              'distractor_signal':[],
              '_original_distractor_timit_indices':[],
              '_original_cue_timit_index':[],
              'distractor_words':[],
              'distractor_speakers':[],
              'distractor_conditions':[],
              'distractor_sex':[],
              'snrs':[]}


for cond_info in tqdm(condition_pairs):
    snr, distractors = cond_info
    n_distractors = len(distractors)
    if distractors == 'ssn':
        condition = distractors
    else:
        condition = len(distractors)

    for ix, item in target_timit.iterrows():
        # add wantned data to trial dict
        update_dict(trial_data, item)
        # Get valid set of distractors 
        valid_distractors = target_timit[target_timit.speaker!=item.speaker].reset_index()
        # Set noise based on condition 
        if condition == 'ssn':
            noise = generate_speech_shaped_noise()
            distractor_indices = 'ssn'
            distractor_words = 'ssn'
            distractor_speakers = 'ssn'
        else:
            if condition == 1:
                distractor_ixs = np.where(valid_distractors.speaker_sex==distractors)[0]
                distractor_ixs = np.random.choice(distractor_ixs, size=1)
                noise = valid_distractors['signal'][distractor_ixs[0]]
            elif condition in [2, 4]:
                talker_sex_counts = dict(zip(*np.unique(distractors, return_counts = True)))
                m_distractors=[]
                f_distractors=[]
                if 'm' in talker_sex_counts:
                    m_distractors = np.where(valid_distractors.speaker_sex =='m')[0]
                    m_distractors = np.random.choice(m_distractors, size=talker_sex_counts['m'])
                if 'f' in talker_sex_counts:
                    f_distractors = np.where(valid_distractors.speaker_sex=='f')[0]
                    f_distractors = np.random.choice(f_distractors, size=talker_sex_counts['f'])
                distractor_ixs = np.concatenate([f_distractors, m_distractors])
                noise = np.stack(valid_distractors['signal'][distractor_ixs]).sum(0) # sum unique distractors 
                
            # get common feats 
            distractor_words = valid_distractors['word'][distractor_ixs].values
            distractor_indices = valid_distractors['_original_timit_index'][distractor_ixs].values
            distractor_speakers = valid_distractors['speaker'][distractor_ixs].values
        
        # create trial stim 
        noise,_ = rms_normalize(noise)
        signal, _ = rms_normalize(item.signal)
        mixture, _ = combine_with_noise(signal, noise, snr) # first_scale_factor unused
        mixture, mixture_scale_factor = rms_normalize(mixture)
        # 
        cue_data = cue_timit[cue_timit.speaker == item.speaker].sample(1)
        cue, _ = rms_normalize(cue_data['signal'].item())
        # rove cue
        cue = cue * mixture_scale_factor
    
        # save values for tiral 
        trial_data['mixture_signal'].append(mixture)
#         trial_data['distractor_signal'].append(noise)
        trial_data['cue_signal'].append(cue)
        trial_data['cue_word'].append(cue_data.word)
        trial_data['cue_speaker'].append(cue_data.speaker)
        trial_data['_original_cue_timit_index'].append(cue_data._original_timit_index)
        trial_data['_original_distractor_timit_indices'].append(distractor_indices)
        trial_data['distractor_words'].append(distractor_words)
        trial_data['distractor_speakers'].append(distractor_speakers)
        trial_data['distractor_conditions'].append(condition)
        trial_data['distractor_sex'].append(''.join(str(token) for token in distractors))
        trial_data['snrs'].append(snr)


    

100%|██████████| 46/46 [06:51<00:00,  8.94s/it]


In [34]:
del trial_data

In [33]:
model_timit_df.cue_speaker[0].item()

'fdaw0'

In [27]:
model_timit_df = pd.DataFrame(trial_data)

In [36]:
model_timit_df['cue_speaker'] = model_timit_df['cue_speaker'].apply(lambda x: x.item())

In [38]:
model_timit_df['cue_word'] = model_timit_df['cue_word'].apply(lambda x: x.item())

In [40]:
model_timit_df['_original_cue_timit_index'] = model_timit_df['_original_cue_timit_index'].apply(lambda x: x.item())

In [54]:
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/End-to-end-ASR-Pytorch/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_and_speaker_encodings['word_idx_to_word']
word_to_ix_map = {val:key for key,val in class_map.items()}



In [56]:
def get_ix_from_words(words):
    if not isinstance(words, list):
        return words
    else:
        return [word_to_ix_map[word] for word in words]

model_timit_df['distractor_word_ints'] = model_timit_df['distractor_words'].apply(get_ix_from_words)


In [57]:
model_timit_df.columns

Index(['signal', 'speaker', 'speaker_sex', 'sentence_type', 'word_int',
       'mixture_signal', 'cue_signal', 'cue_speaker', 'cue_word',
       '_original_distractor_timit_indices', '_original_cue_timit_index',
       'distractor_words', 'distractor_speakers', 'distractor_conditions',
       'distractor_sex', 'snrs', 'word', '_original_timit_index', 'source',
       'sr', 'signal_length', 'sentence_id', 'dialect_region', 'data_split',
       'distractor_word_ints'],
      dtype='object')

In [58]:
out_path = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/')

model_timit_df.to_pickle(out_path / 'timit_attn_stim_for_model_all_targets.pdpkl')

In [3]:
f_path = '/om2/user/imgriff/datasets/timit/attn_task_dataframes/timit_attn_stim_for_model_all_targets.pdpkl'
timit_df = pd.read_pickle(f_path)


In [4]:
timit_df.columns

Index(['signal', 'speaker', 'speaker_sex', 'sentence_type', 'word_int',
       'mixture_signal', 'cue_signal', 'cue_speaker', 'cue_word',
       '_original_distractor_timit_indices', '_original_cue_timit_index',
       'distractor_words', 'distractor_speakers', 'distractor_conditions',
       'distractor_sex', 'snrs', 'word', '_original_timit_index', 'source',
       'sr', 'signal_length', 'sentence_id', 'dialect_region', 'data_split',
       'distractor_word_ints'],
      dtype='object')

In [5]:
from IPython.display import Audio


In [6]:
ix = 0
sample_rate = 20_000
Audio(timit_df['signal'][ix], rate=sample_rate, normalize=False)

In [8]:
Audio(timit_df['cue_signal'][ix], rate=sample_rate, normalize=False)

In [9]:
Audio(timit_df['mixture_signal'][ix], rate=sample_rate, normalize=False)

In [12]:
rms = lambda x: np.sqrt(np.mean(np.square(x)))

In [13]:
rms(timit_df['cue_signal'][ix])

0.04473304400332899

In [14]:
rms(timit_df['mixture_signal'][ix])

0.1